<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Decision Trees and Random Forests

*Session 5 · Notebook 03.04 · Lecture · Coach version*

## Overview

This notebook covers two of the most useful classifiers in risk work. A **decision tree** is a sequence of learned yes/no rules, easy to read and explain. A **random forest** is an ensemble of many trees that trades some of that interpretability for stronger, more stable performance. We train and evaluate both, visualise how a tree makes decisions, read feature importances, and compare the two models on the same split.

We use the Iris dataset (predicting the flower species from four measurements): small, clean and multiclass, so the mechanics are easy to see.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how a decision tree splits data, and why depth controls overfitting.
- Train, evaluate and visualise a decision tree (`plot_tree`).
- Explain how a random forest reduces variance by combining many trees.
- Read feature importances and class probabilities.
- Compare a single tree against a forest and discuss the interpretability trade-off.

## Prerequisites

- Session 5 notebooks 01.02 (the scikit-learn workflow) and 03.02 (classification evaluation with KNN).
- Comfort with `train_test_split` and the confusion matrix / classification report.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is a decision tree?](#sec2)
3. [What is a random forest?](#sec3)
4. [Load and prepare the data](#sec4)
5. [Visualise the class structure](#sec5)
6. [Train and evaluate a decision tree](#sec6)
7. [Explain the decision tree](#sec7)
8. [Train and evaluate a random forest](#sec8)
9. [Feature importance and class probabilities](#sec9)
10. [Compare the models](#sec10)
11. [Exercises](#exercises)
12. [Challenge](#challenge)
13. [Key Takeaways](#takeaways)
14. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

We use `iris_Random_Forest_Dataset.csv` (150 flowers, four measurements and a `Species` label), read from the repo-root `datasets/` folder (two levels up).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Or read directly from the public S3 bucket (no local file needed):
# df = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/iris_Random_Forest_Dataset.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# df = pd.read_csv(session_datasets_http["iris_Random_Forest_Dataset"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# df = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/iris_Random_Forest_Dataset.csv", header=True, inferSchema=True).toPandas()
df = pd.read_csv('../../datasets/Session_5/iris_Random_Forest_Dataset.csv')
print('shape:', df.shape)
df.head()

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Trees and forests are workhorses of credit and fraud modelling.

| Idea | Why a risk team cares |
|---|---|
| A tree is a set of readable rules | It reads like a decision chart or scorecard, which is easy to explain to a regulator or a credit committee |
| Feature importance | Shows which drivers the model relied on (income, history, ...), useful for governance |
| Random forest performance | Captures interactions and non-linear effects that a linear scorecard misses |
| No scaling needed | Trees split on thresholds, so features do not need standardising (unlike KNN) |
| Interpretability vs power | A single tree is transparent; a forest is stronger but harder to explain, a real governance trade-off |


<a id="sec2"></a>
# Section 2: What is a decision tree?

**Definition:** a decision tree classifies a record by asking a sequence of yes/no questions about its features, following the branches until it reaches a leaf that gives the predicted class.

**Example:** "Is petal length below 2.5 cm? If yes, predict setosa. If no, is petal width below 1.7 cm? ...".

**Analogy:** a game of twenty questions, or a doctor's diagnostic flowchart: each answer narrows down the possibilities.

**Explanation:**

- At each node the tree picks the split that best separates the classes, measured by **impurity** (Gini or entropy).
- **Depth controls complexity.** A deep tree can keep splitting until it memorises the training data (**overfitting**); a shallow tree may be too simple (**underfitting**). `max_depth` and `min_samples_leaf` control this.
- Trees need **no feature scaling** (splits are threshold-based) and handle mixed feature types, which makes them convenient.
- They are **highly interpretable**: you can read the rules directly.

**scikit-learn documentation:** [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

### How a decision tree chooses each split: Gini and entropy

**Definition:** at every node the tree tries to make the two child groups as **pure** as possible, meaning each child should hold mostly one class. It measures how mixed a group is with an **impurity** score and picks the feature and threshold that reduce impurity the most. The two common scores are:

- **Gini impurity:** `1 - sum(p_k^2)`, where `p_k` is the fraction of class `k` in the node. It is the chance of mislabelling a random item if you guessed using the node's class proportions.
- **Entropy:** `- sum(p_k * log2(p_k))`, the amount of disorder or uncertainty in the node, measured in bits.

Both are **0 when the node is pure** (a single class) and **largest when the classes are evenly mixed**. For a two-class node they behave almost identically; Gini is scikit-learn's default because it is slightly cheaper to compute.

**Example:** a node of 10 loans that are all 'repaid' has impurity 0 (perfectly pure). A node of 5 'repaid' and 5 'default' is maximally impure (Gini 0.5, entropy 1.0).

**Analogy:** sorting a mixed bag of red and blue marbles into two boxes. A cut is good if each box ends up almost all one colour. Impurity measures how mixed a box still is, and the tree keeps choosing the cut that leaves the tidiest boxes.

**Explanation:** to score a candidate split the tree computes the impurity of each child and takes their **weighted average** (by how many samples land in each child). The drop from the parent's impurity to that weighted-child impurity is the **information gain**. The tree scans every feature and every threshold and keeps the split with the biggest gain. The next two cells show this concretely.

In [ ]:
# Impurity of a node depends only on its class mix
def gini(y):
    p = np.bincount(y) / len(y)
    return 1 - np.sum(p ** 2)

def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]                               # drop zero probabilities before log
    return float(abs(-np.sum(p * np.log2(p))))  # abs avoids a -0.0 display for a pure node

for label, node in [('pure (all one class)', [0] * 10),
                    ('mostly one class (9:1)', [0] * 9 + [1]),
                    ('evenly mixed (5:5)', [0] * 5 + [1] * 5)]:
    arr = np.array(node)
    print(f'{label:26s} Gini = {gini(arr):.3f}   Entropy = {entropy(arr):.3f}')
print('\nImpurity is 0 for a pure node and peaks when the classes are evenly mixed.')

In [ ]:
# 'Across the features': which feature gives the best FIRST split? (a small credit example)
demo = pd.DataFrame({
    'credit_score':  [610, 640, 655, 680, 700, 710, 720, 745, 760, 800],
    'late_payments': [  4,   3,   5,   2,   1,   3,   0,   1,   0,   0],
    'default':       [  1,   1,   1,   1,   0,   1,   0,   0,   0,   0],
})
yv = demo['default'].values
root_gini = gini(yv)
print(f'Root node: {len(yv)} loans, {yv.sum()} defaults, Gini = {root_gini:.3f}\n')

def best_split(feature):
    values = np.sort(demo[feature].unique())
    thresholds = (values[:-1] + values[1:]) / 2          # midpoints between observed values
    best_t, best_gain = None, -1
    for t in thresholds:
        left, right = yv[demo[feature] <= t], yv[demo[feature] > t]
        weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(yv)
        gain = root_gini - weighted                      # information gain
        if gain > best_gain:
            best_t, best_gain = t, gain
    return best_t, best_gain

for feat in ['credit_score', 'late_payments']:
    t, gain = best_split(feat)
    print(f'{feat:14s}: best split at {t:>6.1f}  ->  information gain {gain:.3f}')
print('\nThe tree picks the feature with the largest gain to split on first. Here late_payments '
      'wins, so the root node would split on late_payments. This is exactly what '
      'DecisionTreeClassifier does automatically, across all features, at every node.')

<a id="sec3"></a>
# Section 3: What is a random forest?

**Definition:** a random forest is an **ensemble** of many decision trees. Each tree is trained on a random bootstrap sample of the rows and considers a random subset of features at each split; the forest predicts by majority vote across all the trees.

**Example:** 200 trees each vote on the species of a flower, and the most-voted class wins.

**Analogy:** the wisdom of a crowd. One expert can be biased or wrong; a diverse crowd of experts, voting, is usually more reliable than any single member.

**Explanation:**

- Building each tree on different data and features makes the trees **diverse**; averaging them **reduces variance**, so a forest overfits far less than a single deep tree.
- It is usually **more accurate and more stable** than one tree, at the cost of **interpretability** (you cannot read 200 trees), though feature importances still summarise what mattered.
- Key settings: `n_estimators` (number of trees) and the same depth/leaf controls as a tree.

**scikit-learn documentation:** [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

### The two sources of randomness: bootstrap rows and random feature subsets

A forest beats a single tree only because its trees **disagree** in useful ways. That diversity comes from two deliberate injections of randomness.

**Definition:**

- **Bootstrap sampling (of rows):** each tree is trained not on the full dataset but on a **bootstrap sample**, drawn by picking rows *with replacement* until there are as many rows as the original. Some rows appear several times, others not at all (on average about 37% are left out, the **out-of-bag** rows).
- **Random feature subsets (at each split):** when a tree looks for its best split it does not consider every feature, only a **random subset** (often about the square root of the number of features). Different nodes and different trees therefore focus on different features.

**Example:** with 6 features, each split might only be allowed to choose among a random 2 of them, and each of the 200 trees sees a different resampled version of the rows.

**Analogy:** asking many analysts to each review a slightly different sample of past cases, and forbidding each from leaning on the same few variables. Their individual quirks cancel out when you pool their votes.

**Explanation:** if every tree saw the same rows and all features they would be nearly identical, and averaging them would achieve nothing. By varying the rows (bootstrap) and the features (random subsets), the trees make *different* mistakes, and averaging many different mistakes cancels them out. That is why a forest **reduces variance** and overfits far less than one deep tree. The next cell makes both kinds of randomness visible.

In [ ]:
rng = np.random.default_rng(0)
row_ids = np.arange(10)                                  # pretend the training set has 10 rows (0-9)
features = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6']          # and 6 features

print('Bootstrap samples (rows drawn WITH replacement, so some repeat and some are unused):')
for tree in range(3):
    sample = rng.choice(row_ids, size=len(row_ids), replace=True)
    oob = sorted(int(x) for x in set(row_ids) - set(sample))
    print(f'  Tree {tree + 1}: rows {sorted(sample.tolist())}  |  out-of-bag: {oob}')

# Across a larger sample, about 63% of rows appear and about 37% are out-of-bag
big = rng.choice(np.arange(1000), size=1000, replace=True)
print(f'\nDrawing 1000 rows with replacement used {len(set(big))} distinct rows '
      f'(about {len(set(big)) / 10:.0f}%); the rest are out-of-bag (about 37%).')

k = int(np.sqrt(len(features)))
print(f'\nRandom feature subset at each split (sqrt of {len(features)} features = {k}):')
for split in range(4):
    subset = sorted(rng.choice(features, size=k, replace=False).tolist())
    print(f'  Split {split + 1} may choose only among: {subset}')
print('\nEvery tree sees different rows, and every split sees different features. That diversity '
      'is what makes averaging the trees reduce variance.')

<a id="sec4"></a>
# Section 4: Load and prepare the data

`Id` is just a row identifier, not a measurement, so we drop it. The target is `Species` and the four measurement columns are the features. Because trees do not need scaling, there is no preprocessing pipeline here, just a clean feature/target split.

In [ ]:
df_model = df.drop(columns='Id')
feature_columns = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
X = df_model[feature_columns]
y = df_model['Species']

print('Features:', feature_columns)
print('Classes:', sorted(y.unique()))
print('Class balance:')
print(y.value_counts())

<a id="sec5"></a>
# Section 5: Visualise the class structure

A quick pairplot shows how separable the species are on each pair of measurements. Setosa is clearly distinct; versicolor and virginica overlap a little, which is where a model earns its keep.

In [ ]:
sns.pairplot(df_model, hue='Species', height=1.8)
plt.show()

<a id="sec6"></a>
# Section 6: Train and evaluate a decision tree

We split the data (stratified, to keep the three classes balanced across train and test), then train a shallow tree (`max_depth=3`) so it stays readable. We report **training and test accuracy together**: a big gap between them is the classic sign of overfitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

tree_model = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

tree_test_pred = tree_model.predict(X_test)
print('Training accuracy:', round(tree_model.score(X_train, y_train), 3))
print('Test accuracy    :', round(tree_model.score(X_test, y_test), 3))
print('\nClassification report (test):')
print(classification_report(y_test, tree_test_pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, tree_test_pred, xticks_rotation=30, cmap='Blues')
plt.title('Decision tree confusion matrix'); plt.show()

<a id="sec7"></a>
# Section 7: Explain the decision tree

Two complementary views. **Feature importance** ranks how much each feature contributed to the splits. The **tree diagram** shows the actual rules, which is why trees are prized for interpretability.

In [ ]:
tree_importance = pd.Series(tree_model.feature_importances_, index=feature_columns).sort_values()
tree_importance.plot(kind='barh', figsize=(8, 4), color='steelblue')
plt.title('Decision tree feature importance'); plt.xlabel('relative importance'); plt.show()

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(tree_model, feature_names=feature_columns, class_names=tree_model.classes_,
          filled=True, rounded=True)
plt.title('Decision tree structure'); plt.show()

<a id="sec8"></a>
# Section 8: Train and evaluate a random forest

Now a random forest of 200 trees, evaluated the same way on the same split so the comparison is fair. Left unconstrained (`max_depth=None`), the individual trees can grow deep, but the averaging keeps the forest from overfitting the way a single deep tree would.

In [ ]:
forest_model = RandomForestClassifier(n_estimators=200, max_depth=None,
                                      random_state=RANDOM_STATE, n_jobs=1)
forest_model.fit(X_train, y_train)

forest_test_pred = forest_model.predict(X_test)
print('Training accuracy:', round(forest_model.score(X_train, y_train), 3))
print('Test accuracy    :', round(forest_model.score(X_test, y_test), 3))
print('\nClassification report (test):')
print(classification_report(y_test, forest_test_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, forest_test_pred, xticks_rotation=30, cmap='Blues')
plt.title('Random forest confusion matrix'); plt.show()

<a id="sec9"></a>
# Section 9: Feature importance and class probabilities

The forest also reports feature importances (averaged over all its trees), and, like most classifiers, it can output a **probability** per class with `predict_proba`, not just a hard label. Probabilities are valuable in risk work, where you often want a score, not a yes/no.

In [ ]:
forest_importance = pd.Series(forest_model.feature_importances_, index=feature_columns).sort_values()
forest_importance.plot(kind='barh', figsize=(8, 4), color='seagreen')
plt.title('Random forest feature importance'); plt.xlabel('relative importance'); plt.show()

In [ ]:
proba_preview = pd.DataFrame(forest_model.predict_proba(X_test.head(8)),
                            columns=forest_model.classes_, index=X_test.head(8).index)
proba_preview.round(3)

<a id="sec10"></a>
# Section 10: Compare the models

We compare the two models on the same test set. A single tree is easier to interpret; a random forest is usually at least as accurate and more stable. In practice the choice is not only about accuracy: it also depends on interpretability, stability, data size and deployment constraints (a regulated scorecard may favour the transparent tree).

In [ ]:
from sklearn.model_selection import cross_val_score

cv_tree = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
                          X, y, cv=5).mean()
cv_forest = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                                   n_jobs=1), X, y, cv=5).mean()

comparison = pd.DataFrame({
    'model': ['Decision tree (depth 3)', 'Random forest (200 trees)'],
    'train_accuracy': [tree_model.score(X_train, y_train), forest_model.score(X_train, y_train)],
    'test_accuracy': [tree_model.score(X_test, y_test), forest_model.score(X_test, y_test)],
    'cv_accuracy_5fold': [cv_tree, cv_forest],
}).round(3)
comparison

### Reading the comparison honestly

On this particular test set the shallow tree actually scored a little **higher** than the forest. That is not a contradiction: the test set is tiny (30 rows), so a single split is noisy, and Iris is so easy that a simple tree is already near-optimal. The **cross-validated** column (averaged over 5 folds) is the fairer comparison, and on it the two models are very close. Two lessons: judge models with cross-validation rather than one split, and remember that more complexity is not automatically better.

<a id="exercises"></a>
# Section 11: Exercises

### Exercise 1: A deeper tree

Train a decision tree with `max_depth=1` (a 'stump'), fit it, and print its training and test accuracy. Is it under-fitting compared with the depth-3 tree?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump.fit(X_train, y_train)
print('Stump train accuracy:', round(stump.score(X_train, y_train), 3))
print('Stump test accuracy :', round(stump.score(X_test, y_test), 3))
print('A single split cannot separate three classes, so it under-fits.')

### Exercise 2: Fewer trees

Train a random forest with only `n_estimators=10` and compare its test accuracy to the 200-tree forest. Does having more trees help here?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
small_forest = RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE, n_jobs=1)
small_forest.fit(X_train, y_train)
print('10-tree forest test accuracy :', round(small_forest.score(X_test, y_test), 3))
print('200-tree forest test accuracy:', round(forest_model.score(X_test, y_test), 3))

### Exercise 3: Read feature importance

Print the random forest's feature importances as a sorted table (most important first). Which measurement drives the predictions most?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
imp = pd.Series(forest_model.feature_importances_, index=feature_columns).sort_values(ascending=False)
print(imp.round(3))
print('Petal measurements dominate; they separate the species best.')

<a id="challenge"></a>
## Challenge (optional): watch a tree overfit

Show overfitting directly. Train decision trees for `max_depth` from 1 to 10, record the training and test accuracy for each, and plot both curves on one chart. Explain what happens to the gap between the two curves as the tree gets deeper, and where a forest sits.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
depths = range(1, 11)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.figure(figsize=(9, 5))
plt.plot(list(depths), train_acc, marker='o', label='training accuracy')
plt.plot(list(depths), test_acc, marker='o', label='test accuracy')
plt.axhline(forest_model.score(X_test, y_test), color='green', ls='--',
            label='random forest test accuracy')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend()
plt.title('Decision tree: training vs test accuracy as depth grows'); plt.show()

print('As depth grows the training accuracy climbs towards 1.0 (the tree memorises the data), '
      'but the test accuracy plateaus or dips: the widening gap is overfitting. The random '
      'forest reaches a similar or better test accuracy without that fragility.')

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Decision tree | A flowchart of learned yes/no rules; interpretable |
| `DecisionTreeClassifier(max_depth=...)` | Depth controls complexity; too deep overfits |
| `plot_tree(...)` | Draw the tree's rules |
| `feature_importances_` | Which features drove the splits (not proof of causation) |
| Random forest | Ensemble of many trees; majority vote reduces variance |
| `RandomForestClassifier(n_estimators=...)` | More trees, more stability |
| `predict_proba` | Class probabilities, not just a hard label |
| No scaling needed | Trees split on thresholds, unlike KNN |
| Train vs test accuracy gap | The signature of overfitting |


## Conclusion

You can now train, evaluate, visualise and explain a decision tree, scale that up to a random forest, read feature importances and class probabilities, and reason about the accuracy vs interpretability trade-off. Together with KNN, these give you a solid classification toolkit; logistic regression (the classic scorecard model) is covered separately.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html) how trees split and their parameters.
- [scikit-learn: Ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html) random forests and beyond.
- [scikit-learn: permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html) a more robust importance measure.